# AgentCore Memory를 사용하는 LlamaIndex - 투자 Portfolio Advisor(장기 메모리)

## 소개

이 Notebook에서는 Amazon Bedrock AgentCore Memory 기능을 LlamaIndex와 통합하여 여러 고객 회의와 시장 cycle에 걸쳐 **장기 메모리**를 유지하는 투자 Portfolio Advisor를 만드는 방법을 살펴봅니다. 이를 통해 advisor는 수개월 또는 수년에 걸쳐 투자 지식을 축적하고 portfolio 성과를 추적할 수 있습니다.

## 아키텍처 개요

![LlamaIndex AgentCore 장기 메모리 아키텍처](LlamaIndex-AgentCore-LTM-Arch.png)

## 튜토리얼 세부 정보

**튜토리얼 세부 정보:**
- **튜토리얼 유형**: Session 간 장기 메모리
- **Agent 사용 사례**: 투자 Portfolio Advisor
- **Agentic Framework**: LlamaIndex
- **LLM model**: Anthropic Claude 3.7 Sonnet
- **튜토리얼 구성 요소**: AgentCore 장기 메모리, LlamaIndex Agent, 금융 Tool
- **예제 난이도**: 고급

## 비즈니스 가치

**Enterprise Investment Intelligence**: Portfolio 지식을 축적하고 투자 변화 과정을 추적하며 여러 분기와 연도에 걸쳐 종합적인 시장 분석을 유지하는 지속형 AI 메모리로 자산 관리 업무를 혁신합니다.

**주요 전문적 이점:**
- **Portfolio 연속성**: 투자 기간과 팀 구성원 간에 지식을 원활하게 이전
- **투자 메모리**: 중요한 시장 insight, strategy, 성과 데이터를 영구 보존
- **Portfolio 간 Intelligence**: 여러 고객 portfolio에서 pattern과 연관성 식별
- **전략적 우수성**: 과거 성과 데이터를 활용하여 더 나은 투자 결정
- **고객 관계**: 여러 해에 걸친 자산 관리의 상세 맥락 유지
- **위험 관리**: 시장 cycle과 투자 strategy에 미치는 영향 추적

## 장기 메모리 구성

**기술 설정**: 이 튜토리얼에서는 Semantic Strategy가 적용된 AgentCore Memory를 사용하여 데이터를 12개월간 보존합니다.
- **Memory 유형**: Insight를 자동으로 추출하는 semantic strategy
- **보존 기간**: Portfolio 연속성을 위한 365일 event 만료 기간
- **Session 간 구성**: 동일한 actor_id + memory_id, 투자 기간별로 서로 다른 session_id
- **검색 기능**: 전체 portfolio 기록을 semantic search하는 기본 제공 memory 검색 tool

## 기술 개요

**주요 장기 메모리 구성 요소:**
1. **Semantic Strategy 구성**: SemanticStrategy를 사용하여 insight를 자동 추출하고 365일간 보존
2. **Session 간 지속성**: 동일한 actor_id + memory_id와 기간별로 다른 session_id를 사용하여 지식 연속성 구현
3. **Custom Memory 검색 Tool**: AgentCore 기본 search_long_term_memories()를 LlamaIndex FunctionTool로 wrapping
4. **Semantic 처리 Pipeline**: 대화 event를 semantic memory로 변환하기 위해 90초 대기
5. **동적 Session 관리**: 유연한 session 처리를 위해 memory.context.session_id 사용

**다음 내용을 학습합니다:**

- 여러 고객 회의에 걸쳐 지속되는 AgentCore Memory 생성
- 시간에 따라 투자 지식 축적
- 시장 조사와 고객 기록을 대상으로 semantic search 구현
- Portfolio 변화와 투자 성과 추적
- Session 간 금융 지식 지속성 및 검색 테스트

## 시나리오 배경

이 예제에서는 여러 분기와 연도에 걸친 고객 회의에서 투자 지식을 유지하는 "Investment Portfolio Advisor"를 만듭니다. Advisor는 AgentCore Memory를 사용하여 고객 profile, portfolio 성과, 시장 insight, 투자 결과에 관한 지속형 지식 base를 구축합니다. 이 지식은 시간에 따라 축적되고 발전하여 정교한 장기 자산 관리를 지원합니다.

## 사전 요구 사항

- Python 3.10+
- 적절한 권한이 있는 AWS account
- AgentCore Memory 권한이 있는 AWS IAM role:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock model에 대한 액세스

## 1단계: Dependency 설치 및 설정

In [ ]:
# Semantic strategy toolkit을 포함한 필수 library 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3 bedrock-agentcore-starter-toolkit

In [ ]:
# 필요한 component import
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies.semantic import (
    SemanticStrategy,
)
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

print("✅ All dependencies imported successfully!")

## 2단계: AgentCore Memory 구성

장기 투자 지식을 위한 AgentCore Memory resource를 생성하거나 가져옵니다.

In [ ]:
# 장기 지속성을 위해 Semantic Strategy가 적용된 AgentCore Memory 생성
region = os.getenv("AWS_REGION", "us-east-1")
memory_manager = MemoryManager(region_name=region)

try:
    # Insight 자동 추출용 semantic strategy로 memory 생성
    memory = memory_manager.get_or_create_memory(
        name=f"InvestmentAdvisorSemantic_{int(datetime.now().timestamp())}",
        strategies=[SemanticStrategy(name="investmentLongTermMemory")],
        event_expiry_days=365,  # 투자 record를 12개월간 보존
    )
    memory_id = memory.get("id")
    print(f"✅ Created Semantic Memory: {memory_id}")
    print(f"   Status: {memory.get('status')}")
    print(f"   Strategies: {[s.get('name') if isinstance(s, dict) else str(s) for s in memory.get('strategies', [])]}")

    # Memory가 ACTIVE 상태가 될 때까지 대기
    if memory.get("status") != "ACTIVE":
        print(f"\n⏳ Waiting for memory to become ACTIVE (currently {memory.get('status')})...")
        import time

        max_wait = 300  # 최대 5분
        waited = 0
        while waited < max_wait:
            time.sleep(10)
            waited += 10
            # 상태 확인
            current_memory = memory_manager.get_memory(memory_id)
            status = current_memory.get("status")
            print(f"   [{waited}s] Status: {status}")
            if status == "ACTIVE":
                print(f"✅ Memory is now ACTIVE! (took {waited} seconds)")
                break
        else:
            print(f"⚠️  Memory still not ACTIVE after {max_wait}s. Proceeding anyway...")

except Exception as e:
    print(f"❌ Error creating memory: {e}")
    memory_id = "your-memory-id-here"  # 기존 memory ID로 교체

## 3단계: 투자 Tool 구현

장기 자산 관리를 위한 전문 tool을 정의합니다.

In [ ]:
def record_client_meeting(client_id: str, meeting_type: str, portfolio_value: str, key_decisions: str) -> str:
    """Record client meeting with portfolio updates and decisions"""
    return f"📅 Recorded {meeting_type} for {client_id} (${portfolio_value})"


def track_portfolio_performance(
    client_id: str,
    period: str,
    return_pct: str,
    benchmark_return: str,
    attribution: str,
) -> str:
    """Track portfolio performance vs benchmark with attribution analysis"""
    return f"📈 {client_id} {period}: {return_pct} vs {benchmark_return}"


def document_market_insight(
    insight_type: str,
    market_event: str,
    impact_assessment: str,
    client_implications: str,
) -> str:
    """Document market insight with client portfolio implications"""
    print(f"🌍 Market insight: {insight_type} - {market_event} (Impact: {impact_assessment})")
    return f"Documented market insight: {insight_type}"


def update_investment_thesis(client_id: str, asset_class: str, thesis: str, conviction_level: str) -> str:
    """Update investment thesis for specific asset class"""
    print(f"💭 Investment thesis: {client_id} - {asset_class} ({conviction_level} conviction)")
    return f"Updated thesis for {client_id}"


def log_rebalancing_action(client_id: str, action_type: str, securities: str, rationale: str) -> str:
    """Log portfolio rebalancing actions with rationale"""
    print(f"⚖️ Rebalancing: {client_id} - {action_type}: {securities}")
    return f"Logged rebalancing for {client_id}"


def log_advisory_milestone(quarter: str, milestone: str, details: str) -> str:
    """Log an advisory milestone with quarter and detailed progress"""
    print(f"🎯 {quarter} milestone: {milestone}")
    return f"Logged milestone for {quarter}: {milestone} - {details}"


def track_investment_metrics(metric_type: str, value: str, client_id: str, quarter: str) -> str:
    """Track specific investment metrics with client and timeline"""
    print(f"📊 {quarter}: {metric_type} = {value} (for {client_id})")
    return f"Tracked {metric_type}: {value} for {client_id} in {quarter}"


def save_advisory_insight(insight: str, quarter: str, market_context: str) -> str:
    """Save advisory insights with market context"""
    print(f"💡 {quarter} insight: {insight[:50]}...")
    return f"Saved {quarter} insight with market context: {market_context}"


# Agent용 tool object 생성
investment_tools = [
    FunctionTool.from_defaults(fn=record_client_meeting),
    FunctionTool.from_defaults(fn=track_portfolio_performance),
    FunctionTool.from_defaults(fn=document_market_insight),
    FunctionTool.from_defaults(fn=update_investment_thesis),
    FunctionTool.from_defaults(fn=log_rebalancing_action),
    FunctionTool.from_defaults(fn=log_advisory_milestone),
    FunctionTool.from_defaults(fn=track_investment_metrics),
    FunctionTool.from_defaults(fn=save_advisory_insight),
]

print("✅ Investment tools created!")

## 3b단계: Memory 검색 Tool 추가

Agent가 장기 메모리를 검색할 수 있는 tool을 생성합니다.

In [ ]:
def create_memory_retrieval_tool(memory_id: str, actor_id: str, region: str):
    """에이전트가 자체 장기 메모리를 검색하는 도구를 생성합니다."""

    def search_long_term_memory(query: str) -> str:
        """Search long-term memory for relevant information about clients, portfolios, past decisions, and market insights.

        Use this tool when you need to recall:
        - Client information (portfolio values, risk profiles, investment goals)
        - Past investment decisions and their outcomes
        - Portfolio performance history
        - Market insights and their applications
        - Investment theses and their evolution

        Args:
            query: Search query describing what information you need (e.g., 'CLIENT-001 portfolio', 'investment theses', 'Q1 performance')

        Returns:
            Relevant information from long-term memory
        """
        try:
            from bedrock_agentcore.memory.session import MemorySessionManager

            # Session manager 생성 (memory_id와 region만 필요)
            session_manager = MemorySessionManager(memory_id=memory_id, region_name=region)

            # Semantic strategy namespace에서 장기 메모리 검색
            results = session_manager.search_long_term_memories(
                query=query,
                namespace_prefix="/strategies/",  # Semantic strategy namespace에서 검색
                top_k=5,
                max_results=10,
            )

            if not results:
                return "No relevant information found in long-term memory. This might be new information or the memory extraction may still be processing."

            # Agent용 결과 형식 지정
            output = "📚 Retrieved from long-term memory:\\n\\n"
            for i, result in enumerate(results, 1):
                # MemoryRecord object의 content attribute에 액세스
                content = getattr(result, "content", str(result))
                # 매우 긴 content 자르기
                if len(content) > 300:
                    content = content[:300] + "..."
                output += f"{i}. {content}\\n\\n"

            return output

        except Exception as e:
            return f"⚠️ Error searching memory: {str(e)}. Proceeding without historical context."

    return FunctionTool.from_defaults(fn=search_long_term_memory)


# Memory 검색 tool 생성
memory_search_tool = create_memory_retrieval_tool(memory_id, "financial-advisor", region)

# Tool 목록에 memory 검색 추가
investment_tools_with_memory = investment_tools + [memory_search_tool]

print(f"✅ Memory retrieval tool created! Total tools: {len(investment_tools_with_memory)}")
print("   Using namespace: /strategies/ (for semantic strategy compatibility)")

## 3c단계: Memory 구성 확인

Semantic strategy가 올바르게 구성되었는지 확인합니다.

In [ ]:
# Memory 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"Strategies: {memory_info.get('strategies')}")
print(f"Status: {memory_info.get('status')}")
print(f"Name: {memory_info.get('name')}")

# Strategy 세부 정보 표시
strategies = memory_info.get("strategies", [])
for strategy in strategies:
    print("\nStrategy Details:")
    print(f"  Name: {strategy.get('name')}")
    print(f"  Type: {strategy.get('type')}")
    print(f"  Status: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 4단계: Multi-Session Agent 구현

서로 다른 advisory 기간을 시뮬레이션하는 helper function을 생성합니다.

In [ ]:
# 장기 메모리 구성 (session 간)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
ADVISOR_ID = "financial-advisor"  # 모든 session에서 동일한 advisor


def create_advisory_session(session_name: str):
    """장기 메모리가 유지되는 새 자문 세션을 생성합니다."""
    context = AgentCoreMemoryContext(
        actor_id=ADVISOR_ID,  # 동일한 advisor
        memory_id=memory_id,  # 동일한 memory store (장기 메모리 활성화)
        session_id=f"advisory-{session_name}",  # 기간별로 다른 session
        namespace="/wealth-management/",
    )

    memory = AgentCoreMemory(context=context)
    llm = BedrockConverse(model=MODEL_ID)
    agent = FunctionAgent(
        tools=investment_tools_with_memory,  # Memory 검색 기능이 있는 tool 사용
        llm=llm,
        verbose=True,  # Memory 검색 시점을 확인하도록 verbose 활성화
        system_prompt="""You are a senior investment advisor with access to long-term memory.
        
CRITICAL: When asked about clients, portfolios, past decisions, or historical information, 
you MUST use the search_long_term_memory tool FIRST before responding.

For example:
- "What clients am I managing?" → Use search_long_term_memory("clients portfolio")
- "What was CLIENT-001's performance?" → Use search_long_term_memory("CLIENT-001 performance")
- "What investment theses do I have?" → Use search_long_term_memory("investment thesis")

Always provide conclusive, complete responses without asking follow-up questions.\n
Execute all requested actions immediately and completely. Provide detailed, professional responses.""",
    )

    return agent, memory


print("✅ Multi-session Investment Portfolio Advisor setup complete!")

## 5단계: Q1 Advisory Session - 초기 고객 Onboarding

첫 advisory session을 시작하고 고객 baseline을 설정합니다.

In [ ]:
# === Q1 ADVISORY SESSION 시작 ===
print("🗓️ === Q1: INITIAL CLIENT ONBOARDING ===")

agent_q1, memory_q1 = create_advisory_session("q1")

# 초기 고객 회의 기록
response = await agent_q1.run(
    "I'm Senior Advisor Jennifer Walsh. Record client meeting for 'CLIENT-001' with meeting type 'Initial Portfolio Review', "
    "portfolio value '$3,200,000', key decisions 'established moderate-aggressive risk profile, 20-year investment horizon, "
    "target allocation 70% equity/25% fixed income/5% alternatives'.",
    memory=memory_q1,
)

print("🎯 Q1 Initial Meeting:")
print(response)

In [ ]:
# 초기 투자 thesis 문서화
response = await agent_q1.run(
    "Update investment thesis for 'CLIENT-001': asset class 'US Large Cap Equity', "
    "thesis 'overweight growth stocks due to technological innovation and earnings momentum', conviction level 'high'.",
    memory=memory_q1,
)
print("💭 Q1 Equity Thesis:", response)

response = await agent_q1.run(
    "Update investment thesis for 'CLIENT-001': asset class 'Fixed Income', "
    "thesis 'short duration bias due to rising rate environment, focus on credit quality', conviction level 'medium'.",
    memory=memory_q1,
)
print("💭 Q1 Bond Thesis:", response)

In [ ]:
# 초기 성과 baseline 추적
response = await agent_q1.run(
    "Track portfolio performance for 'CLIENT-001': period 'Q1 2024', return_pct '+8.2%', "
    "benchmark_return '+7.1%', attribution 'tech overweight +0.8%, duration underweight +0.3%'.",
    memory=memory_q1,
)
print("📈 Q1 Performance:", response)

# Event 저장 여부 확인
print("\n🔍 Verifying Q1 events were stored...")
try:
    client = MemoryClient(region_name=region)
    events = client.list_events(
        memory_id=memory_id,
        actor_id=ADVISOR_ID,
        session_id=memory_q1.context.session_id,
    )
    print(f"✅ Stored {len(events)} conversational events in Q1 session")
except Exception as e:
    print(f"⚠️  Could not verify events: {e}")

# Semantic memory 처리 시간 확보
import asyncio

print("\n⏳ Waiting for semantic memory extraction and indexing...")
print("   (AgentCore processes conversational events in the background)")
await asyncio.sleep(90)  # Memory 추출 대기 시간을 10초에서 연장
print("✅ Memory processing complete - memories should now be searchable")

## 6단계: Q2 Advisory Session - 시장 변동성 대응

장기 메모리 검색을 테스트하고 시장 변화에 대응합니다.

In [ ]:
# === Q2 ADVISORY SESSION 시작 ===
print("\n🗓️ === Q2: MARKET VOLATILITY RESPONSE (NEW SESSION) ===")

agent_q2, memory_q2 = create_advisory_session("q2")

# Session 간 고객 회상 테스트 - agent가 search_long_term_memory tool을 사용해야 함
print("\n🧠 Testing memory retrieval across sessions...")
print("   (Watch for the agent to use search_long_term_memory tool)\n")

response = await agent_q2.run(
    "What clients am I managing? What are their portfolio values, risk profiles, and investment theses?",
    memory=memory_q2,
)

print("\n🧠 Q2 Client Recall:")
print(response)
print("\n✅ Expected: CLIENT-001, $3.2M portfolio, moderate-aggressive, growth equity thesis")

In [ ]:
# 시장 변동성 insight 문서화
response = await agent_q2.run(
    "Document market insight: insight type 'Geopolitical Risk', market event 'Trade tensions escalation', "
    "impact assessment 'increased volatility, sector rotation from growth to value', "
    "client implications 'review tech overweight, consider defensive positioning'.",
    memory=memory_q2,
)
print("🌍 Q2 Market Insight:", response)

# Rebalancing 대응 기록
response = await agent_q2.run(
    "Log rebalancing action for 'CLIENT-001': action type 'Tactical Adjustment', "
    "securities 'reduced QQQ by 3%, increased VTV (value ETF) by 2%, added VGSH (short treasury) by 1%', "
    "rationale 'defensive positioning due to geopolitical uncertainty, maintain long-term allocation targets'.",
    memory=memory_q2,
)
print("⚖️ Q2 Rebalancing:", response)

In [ ]:
# Q2 성과 영향 추적
response = await agent_q2.run(
    "Track portfolio performance for 'CLIENT-001': period 'Q2 2024', return_pct '-2.1%', "
    "benchmark_return '-3.8%', attribution 'defensive positioning +1.2%, value tilt +0.5%'.",
    memory=memory_q2,
)
print("📈 Q2 Performance:", response)

# 성과 비교 테스트
response = await agent_q2.run(
    "How did CLIENT-001's Q2 performance compare to Q1? What was the cumulative return and attribution?",
    memory=memory_q2,
)
print("📊 Q2 Performance Analysis:")
print(response)
print("\n✅ Expected: Q1 +8.2%, Q2 -2.1%, cumulative ~+5.9%, defensive positioning helped")

## 7단계: Q3 Advisory Session - 회복 및 Thesis 업데이트

시장 회복 단계로 진행하고 투자 접근 방식을 업데이트합니다.

In [ ]:
# === Q3 ADVISORY SESSION 시작 ===
print("\n🗓️ === Q3: MARKET RECOVERY AND THESIS UPDATE ===")

agent_q3, memory_q3 = create_advisory_session("q3")

# 분기 검토 회의 기록
response = await agent_q3.run(
    "Record client meeting for 'CLIENT-001' with meeting type 'Quarterly Review', "
    "portfolio value '$3,450,000', key decisions 'market recovery positioning, increase growth allocation, "
    "add international exposure for diversification'.",
    memory=memory_q3,
)
print("📅 Q3 Quarterly Review:", response)

# 시장 변화에 따라 투자 thesis 업데이트
response = await agent_q3.run(
    "Update investment thesis for 'CLIENT-001': asset class 'International Equity', "
    "thesis 'add developed market exposure via VTIAX, emerging markets recovery potential', conviction level 'medium'.",
    memory=memory_q3,
)
print("💭 Q3 International Thesis:", response)

In [ ]:
# 종합적인 투자 기록 회상 테스트
response = await agent_q3.run(
    "What is the complete investment history for CLIENT-001? Include all meetings, performance periods, "
    "rebalancing actions, and evolution of investment theses.",
    memory=memory_q3,
)
print("📋 Q3 Complete History:")
print(response)
print("\n✅ Expected: Q1 onboarding → Q2 defensive moves → Q3 recovery positioning, all performance data")

## 8단계: Q4 Advisory Session - 연말 검토 및 계획

Semantic search와 연간 성과 분석을 테스트합니다.

In [ ]:
# === Q4 ADVISORY SESSION 시작 ===
print("\n🗓️ === Q4: YEAR-END REVIEW AND PLANNING ===")

agent_q4, memory_q4 = create_advisory_session("q4")

# 연간 성과 추적
response = await agent_q4.run(
    "Track portfolio performance for 'CLIENT-001': period '2024 Annual', return_pct '+12.8%', "
    "benchmark_return '+11.2%', attribution 'tactical positioning +1.1%, sector allocation +0.5%'.",
    memory=memory_q4,
)
print("📈 Q4 Annual Performance:", response)

# 시장 insight 상관관계 테스트
response = await agent_q4.run(
    "What market insights have I documented this year? How did they impact CLIENT-001's portfolio decisions?",
    memory=memory_q4,
)
print("🌍 Q4 Market Insight Analysis:")
print(response)
print("\n✅ Expected: Geopolitical risk insight → defensive positioning → outperformance during volatility")

In [ ]:
# 유사한 portfolio 작업의 semantic search 테스트
response = await agent_q4.run(
    "What rebalancing actions have I taken for CLIENT-001? Which were most effective based on subsequent performance?",
    memory=memory_q4,
)
print("⚖️ Q4 Rebalancing Analysis:")
print(response)
print("\n✅ Expected: Q2 defensive moves (QQQ reduction, VTV/VGSH adds) helped during volatility")

## 9단계: 2년 차 Q1 Session - 다년간 관점

장기 투자 지식과 고객 관계의 변화를 테스트합니다.

In [ ]:
# === 2년 차 Q1 ADVISORY SESSION 시작 ===
print("\n🗓️ === YEAR 2 Q1: MULTI-YEAR PERSPECTIVE ===")

agent_y2q1, memory_y2q1 = create_advisory_session("year2-q1")

# 다년간 portfolio 분석
response = await agent_y2q1.run(
    "Analyze CLIENT-001's investment journey: How has their portfolio evolved over the past year? "
    "What were the key decisions and their outcomes?",
    memory=memory_y2q1,
)
print("📊 Year 2 Q1 Journey Analysis:")
print(response)
print("\n✅ Expected: $3.2M → $3.45M growth, defensive positioning success, thesis evolution")

In [ ]:
# 투자 thesis 변화 추적 테스트
response = await agent_y2q1.run(
    "How have my investment theses for CLIENT-001 evolved? What asset classes have I added and why?",
    memory=memory_y2q1,
)
print("💭 Year 2 Q1 Thesis Evolution:")
print(response)
print(
    "\n✅ Expected: Started with US equity/fixed income → added international exposure → evolved based on market conditions"
)

## 10단계: 최종 자산 관리 Portfolio 평가

장기 투자 advisory 기능을 종합적으로 테스트합니다.

In [ ]:
# 최종 종합 자산 관리 portfolio 질의
response = await agent_y2q1.run(
    "Provide my complete wealth management portfolio: all clients with their investment journeys, "
    "performance attribution, market insights applied, rebalancing effectiveness, and thesis evolution. "
    "Include lessons learned and best practices developed.",
    memory=memory_y2q1,
)
print("💼 Complete Wealth Management Portfolio:")
print(response)
print("\n✅ Expected: Full CLIENT-001 journey with performance attribution, market timing, and investment evolution")

## 🧪 자동 테스트 검증
이 셀을 실행하여 메모리 통합이 올바르게 작동하는지 검증합니다.

In [ ]:
# Validation function을 inline으로 정의
class TestValidator:
    def __init__(self):
        self.results = {}

    def validate_memory_recall(self, response):
        """에이전트가 세션 앞부분의 정보를 기억하는지 확인합니다."""
        # "I don't know"만 반환한 것이 아닌 실질적인 응답인지 확인
        has_content = len(response) > 50
        # Memory indicator 확인
        has_memory_indicators = any(
            word in response.lower()
            for word in [
                "earlier",
                "mentioned",
                "discussed",
                "previously",
                "you",
                "we",
                "our",
            ]
        )
        return "✅ PASS" if (has_content and has_memory_indicators) else "❌ FAIL"

    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인합니다."""
        has_memory_content = len(response) > 100 and any(
            word in response.lower()
            for word in [
                "previous",
                "earlier",
                "mentioned",
                "discussed",
                "before",
                "already",
            ]
        )
        return "✅ PASS" if has_memory_content else "❌ FAIL"

    def validate_cross_reference(self, response):
        """에이전트가 현재 질의를 이전 컨텍스트와 연결할 수 있는지 확인합니다."""
        # 연결 표현 확인
        connecting_words = [
            "relate",
            "connection",
            "previous",
            "earlier",
            "discussed",
            "mentioned",
            "context",
            "based on",
            "as we",
            "as i",
        ]
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "✅ PASS" if (has_connection and has_substance) else "❌ FAIL"

    def run_validation_summary(self, test_results):
        print("🧪 COMPREHENSIVE TEST VALIDATION SUMMARY")
        print("=" * 60)

        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "PASS" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0

        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")

        print("=" * 60)
        print(f"📊 Overall Pass Rate: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")

        if pass_rate >= 80:
            print("✅ EXCELLENT: Memory integration working correctly!")
        elif pass_rate >= 60:
            print("⚠️  GOOD: Most memory features working, some issues to investigate")
        else:
            print("❌ NEEDS ATTENTION: Memory integration has significant issues")

        return pass_rate


validator = TestValidator()
print("✅ Validation functions loaded!")

In [ ]:
# 모든 validation test 실행
test_results = {}

# 테스트 1: Memory 회상 - 에이전트가 논의 내용을 기억하는가?
response1 = await agent_y2q1.run("What have we discussed so far in this session?", memory=memory_y2q1)
test_results["Memory Recall"] = validator.validate_memory_recall(str(response1))
print(f"Response 1 length: {len(str(response1))} chars")

# 테스트 2: Session memory - 에이전트가 맥락을 유지하는가?
response2 = await agent_y2q1.run("What did we talk about earlier?", memory=memory_y2q1)
test_results["Session Memory"] = validator.validate_session_memory(str(response2))
print(f"Response 2 length: {len(str(response2))} chars")

# 테스트 3: 상호 참조 기능 - 이전 맥락과 연결할 수 있는가?
response3 = await agent_y2q1.run("How does this relate to what we discussed before?", memory=memory_y2q1)
test_results["Cross Reference"] = validator.validate_cross_reference(str(response3))
print(f"Response 3 length: {len(str(response3))} chars")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 Notebook에서는 다음 내용을 살펴봤습니다.

✅ **장기 메모리 통합**: LlamaIndex와 AgentCore Memory를 사용하여 session 간 자산 관리 구현

✅ **투자 과정 추적**: 여러 분기에 걸친 portfolio 변화와 성과 기여도 추적

✅ **시장 Intelligence**: 시장 insight와 portfolio 적용 사례를 semantic 방식으로 검색

✅ **투자 Thesis 변화**: 초기 positioning에서 시장 적응형 strategy로 자연스럽게 발전

✅ **성과 기여도**: 전술적 결정과 투자 결과를 상세하게 추적

✅ **우수한 자산 관리**: 시간에 따른 종합적인 고객 관계 및 portfolio 최적화

Investment Portfolio Advisor는 장기 메모리를 통해 전체 투자 기록을 유지하고 장기 고객 관계 전반에서 정교한 금융 지식 검색을 지원하며 시간이 지날수록 더 똑똑해지는 지속적인 자산 관리 partner로 발전할 수 있음을 보여 줍니다.

## 정리

이 Notebook에서 사용한 resource를 정리하도록 memory를 삭제하겠습니다.

**참고**: Memory를 영구 삭제하려는 경우에만 실행하세요. memory_id 변수에는 이 Notebook의 앞부분에서 생성한 memory의 ID가 있어야 합니다.

In [ ]:
# AgentCore Memory resource 정리
try:
    from bedrock_agentcore.memory import MemoryClient

    client = MemoryClient(region_name=region)
    client.delete_memory(memory_id)
    print(f"✅ Successfully deleted memory: {memory_id}")

except NameError as e:
    print(f"⚠️  Variable not defined: {e}")
    print("Run the notebook from the beginning or set variables manually:")
    print("# memory_id = 'your-memory-id-here'")
    print("# region = 'us-east-1'")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")